# EML Neural Architecture Search (NAS)

This notebook demonstrates `EMLActivationSearch`: evolutionary search for
activation functions over the EML grammar.

**Key idea**: Every elementary activation (ReLU, GELU, SiLU, ELU) is a
depth-bounded EML tree.  We search the space of all such trees to discover
novel activations tailored to specific tasks.


In [ ]:
import numpy as np
from monogate.nas import EMLActivationSearch, complexity_penalized_fitness, regression_fitness

print("EML NAS ready.")

## 1. Setup — Regression Task

In [ ]:
# Target: x^2 + sin(x) — a classic test for activation discovery
rng = np.random.default_rng(0)
X = np.linspace(-2, 2, 100).reshape(-1, 1)
y = X.ravel() ** 2 + np.sin(X.ravel())

print(f"Dataset: {len(y)} samples, X ∈ [{X.min():.1f}, {X.max():.1f}]")
print(f"Target: x² + sin(x), range [{y.min():.3f}, {y.max():.3f}]")

## 2. Search for Best EML Activation

In [ ]:
nas = EMLActivationSearch(
    max_depth=4,
    population_size=40,
    mutation_rate=0.3,
    crossover_rate=0.2,
    seed=42,
)

print("Starting EML NAS...")
result = nas.search_for_activation(
    X, y,
    n_generations=20,
    alpha=0.01,
    log_every=5,
)

print(f"\n=== NAS Result ===")
print(f"Best formula: {result.best_formula}")
print(f"Fitness (MSE + penalty): {result.fitness:.6f}")
print(f"EML nodes: {result.n_nodes}")
print(f"Found at generation: {result.generation}")
print(f"Search time: {result.elapsed_s:.2f}s")

## 3. Evolution Curve

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    gens, fits = zip(*result.history)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(gens, fits, 'b-o', markersize=4, linewidth=1.5)
    ax.set_xlabel("Generation")
    ax.set_ylabel("Best Fitness")
    ax.set_title(f"EML NAS Evolution\nBest: {result.best_formula}")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/tmp/nas_evolution.png", dpi=100)
    print("Evolution curve saved to /tmp/nas_evolution.png")
    plt.close()
except ImportError:
    print("matplotlib not available")

print("\nEvolution history:")
for gen, fit in result.history:
    print(f"  Gen {gen:2d}: fitness={fit:.6f}")

## 4. Compare Against Baseline Activations

In [ ]:
comparison = nas.compare_activations(result.best_tree, X, y)

print("Activation comparison (MSE on regression task):")
print("-" * 45)
sorted_results = sorted(comparison.items(), key=lambda x: x[1])
for name, mse in sorted_results:
    marker = " ← WINNER" if name == "eml_discovered" else ""
    print(f"  {name:20s}  MSE={mse:.6f}{marker}")

## 5. Activation Shape Analysis

In [ ]:
from monogate.search.mcts import _eval_tree

x_plot = np.linspace(-3, 3, 200)

# Evaluate discovered activation
discovered_vals = []
for xi in x_plot:
    try:
        v = _eval_tree(result.best_tree, xi)
        discovered_vals.append(v if isinstance(v, (int, float)) and abs(v) < 1e6 else float('nan'))
    except Exception:
        discovered_vals.append(float('nan'))

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Activation shape
    ax = axes[0]
    ax.plot(x_plot, discovered_vals, 'b-', linewidth=2, label=f'EML: {result.best_formula[:30]}')
    ax.plot(x_plot, np.maximum(0, x_plot), 'r--', alpha=0.7, label='ReLU')
    ax.plot(x_plot, x_plot / (1 + np.exp(-x_plot)), 'g--', alpha=0.7, label='SiLU')
    ax.set_title("Discovered vs Baseline Activations")
    ax.set_xlabel("x")
    ax.set_ylabel("activation(x)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-5, 10)

    # Fit quality
    ax = axes[1]
    y_pred = np.array(discovered_vals[:len(X)])
    ax.scatter(X.ravel(), y, alpha=0.5, s=10, label='Data')
    ax.scatter(X.ravel(), y_pred, alpha=0.5, s=10, c='red', label='EML fit')
    ax.set_title("EML Activation Fit Quality")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("/tmp/nas_activation.png", dpi=100)
    print("Activation plot saved to /tmp/nas_activation.png")
    plt.close()
except ImportError:
    print("matplotlib not available")

## 6. Theory Notes

The EML grammar generates all elementary activations:

| Activation | EML Representation | Depth |
|------------|-------------------|-------|
| Softplus $\log(1+e^x)$ | `eml(-x, eml(-x, 1))` | 2 |
| Sigmoid $\sigma(x)$ | `1/eml(-x, 1)` | 1 + recip |
| SiLU $x\sigma(x)$ | depth-2 composition | 2 |
| GELU approx | depth-3 composition | 3 |

The NAS discovers task-specific variants of these building blocks,
sometimes finding equivalent simpler expressions or genuinely novel forms.

In [ ]:
# Sanity check: custom fitness function
nas2 = EMLActivationSearch(population_size=20, seed=0)

def abs_fitness(tree):
    return regression_fitness(tree, X.ravel(), np.abs(X.ravel()))

result2 = nas2.search(abs_fitness, n_generations=5, log_every=0)
print(f"\nCustom fitness result: {result2.best_formula}")
print(f"Fitness (MSE vs |x|): {result2.fitness:.6f}")